# 02a — Training YOLO11-Pose (Small / Medium / XLarge)

**Runtime:** GPU T4 (Colab). Estimasi total: ~6–9 jam.

**Yang dilakukan notebook ini:**
1. Install Ultralytics
2. Download pretrained YOLO11-Pose weights
3. Training loop: Small → Medium → XLarge (sequential)
4. Simpan `best.pt` + `train_log.json` ke Google Drive

> 💡 Jalankan Cell 1–4 dulu tanpa GPU untuk verifikasi setup, baru aktifkan GPU untuk Cell 5 (training).

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install ultralytics pyyaml --quiet
print('✅ Dependencies installed')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst. Aman dipanggil berkali-kali (idempotent),
    termasuk kalau dst adalah broken symlink dari sesi sebelumnya."""
    if os.path.exists(src) and not os.path.lexists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    if not os.path.exists(f'{PROJECT_ROOT}/src'):
        raise RuntimeError(
            f'Git clone gagal atau repo kosong: {GITHUB_REPO}\n'
            f'Cek: (1) URL repo benar, (2) repo sudah di-push berisi folder src/,\n'
            f'(3) repo Public atau token akses sudah diset via Kaggle Secrets.'
        )
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

DATASET_YAML = f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/data.yaml'
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Pastikan semua folder output ada (wajib di Kaggle) ───────────────────────
import os
_dirs_to_create = [
    f'{PROJECT_ROOT}/artifacts/results/figures',
    f'{PROJECT_ROOT}/artifacts/results/tables',
    f'{PROJECT_ROOT}/artifacts/results/videos',
    f'{PROJECT_ROOT}/artifacts/logs/evaluation',
    f'{PROJECT_ROOT}/artifacts/logs/training',
    f'{PROJECT_ROOT}/artifacts/weights/yolo11/small',
    f'{PROJECT_ROOT}/artifacts/weights/yolo11/medium',
    f'{PROJECT_ROOT}/artifacts/weights/yolo11/xlarge',
    f'{PROJECT_ROOT}/artifacts/weights/hrnet/w18',
    f'{PROJECT_ROOT}/artifacts/weights/hrnet/w32',
    f'{PROJECT_ROOT}/artifacts/weights/hrnet/w48',
    f'{PROJECT_ROOT}/artifacts/weights/vitpose/small',
    f'{PROJECT_ROOT}/artifacts/weights/vitpose/base',
    f'{PROJECT_ROOT}/artifacts/weights/vitpose/large',
    f'{PROJECT_ROOT}/artifacts/weights/detr/r50',
    f'{PROJECT_ROOT}/artifacts/weights/detr/r101',
    f'{PROJECT_ROOT}/data/converted/annotations',
    f'{PROJECT_ROOT}/data/converted/images/train',
    f'{PROJECT_ROOT}/data/converted/images/val',
    f'{PROJECT_ROOT}/data/converted/images/test',
]
for _d in _dirs_to_create:
    os.makedirs(_d, exist_ok=True)
print('Semua folder output siap')

In [ ]:
# ── Cell 3: Load global config ───────────────────────────────────────────────
import yaml
with open(f'{PROJECT_ROOT}/configs/eval_global.yaml') as f:
    EVAL_CFG = yaml.safe_load(f)

print('eval_global.yaml loaded:')
for k, v in EVAL_CFG.items():
    if not isinstance(v, (dict, list)):
        print(f'  {k}: {v}')

In [ ]:
# ── Cell 4: Verifikasi GPU ────────────────────────────────────────────────────
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Aktifkan GPU di Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 5: Training loop — semua variant ────────────────────────────────────
from ultralytics import YOLO
import gc

VARIANTS = [
    {
        'name'       : 'small',
        'pretrained' : 'yolo11s-pose.pt',
        'batch'      : 32,
    },
    {
        'name'       : 'medium',
        'pretrained' : 'yolo11m-pose.pt',
        'batch'      : 32,
    },
    {
        'name'       : 'xlarge',
        'pretrained' : 'yolo11x-pose.pt',
        'batch'      : 16,   # XLarge lebih boros memory
    },
]

for v in VARIANTS:
    print(f'\n{'='*55}')
    print(f'  Training YOLO11-{v["name"].upper()}')
    print(f'{'='*55}')
    
    output_dir = f'{PROJECT_ROOT}/artifacts/weights/yolo11/{v["name"]}'
    os.makedirs(output_dir, exist_ok=True)
    
    t_start = datetime.now()
    model = YOLO(v['pretrained'])
    
    results = model.train(
        data     = DATASET_YAML,
        epochs   = 500,
        imgsz    = 640,
        batch    = v['batch'],
        optimizer= 'SGD',
        lr0      = 0.01,
        lrf      = 0.01,
        momentum = 0.937,
        mosaic   = 0.0,
        flipud   = 0.0,
        fliplr   = 0.5,
        patience = 50,
        project  = output_dir,
        name     = 'run',
        exist_ok = True,
        device   = 0,
        verbose  = False,
    )
    
    t_end = datetime.now()
    duration = (t_end - t_start).total_seconds() / 3600
    
    # Simpan training log
    log = {
        'model_family'  : 'yolo11',
        'model_variant' : v['name'],
        'trained_at'    : t_start.isoformat(),
        'duration_hours': round(duration, 2),
        'final_mAP50'   : float(results.results_dict.get('metrics/mAP50(P)', 0)),
        'final_mAP5095' : float(results.results_dict.get('metrics/mAP50-95(P)', 0)),
        'best_epoch'    : int(results.best_epoch) if hasattr(results, 'best_epoch') else -1,
        'gpu'           : torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    }
    
    log_path = f'{output_dir}/run/train_log.json'
    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)
    
    print(f'  ✅ Done! Duration: {duration:.1f}h')
    print(f'  mAP@0.5: {log["final_mAP50"]:.4f}')
    print(f'  Log saved: {log_path}')
    
    # Bersihkan memory sebelum variant berikutnya
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  🧹 GPU memory cleared')

print('\n🎉 Semua YOLO11 variants selesai ditraining!')

In [ ]:
# ── Cell 6: Verifikasi weights tersimpan ─────────────────────────────────────
for v in ['small', 'medium', 'xlarge']:
    best_pt = f'{PROJECT_ROOT}/artifacts/weights/yolo11/{v}/run/weights/best.pt'
    if os.path.exists(best_pt):
        size_mb = os.path.getsize(best_pt) / 1024**2
        print(f'  ✅ YOLO11-{v:6s}: {best_pt} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ YOLO11-{v:6s}: best.pt tidak ditemukan!')

print('\n📌 Langkah selanjutnya: notebooks/02b_train_hrnet.ipynb')